In [1]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'

In [ ]:
import shutil
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import subprocess
from pathlib import Path

WORKDIR = Path(os.environ.get('SNU_WORKDIR', '/content/snu_ai'))
DRIVE_ROOT = Path(os.environ.get('SNU_DRIVE_ROOT', '/content/drive/MyDrive/snu ai'))
LOCAL_DATA_ROOT = WORKDIR / 'snuaichallenge_data'
DRIVE_DATA_ROOT = DRIVE_ROOT / 'snuaichallenge_data'

def has_complete_data(root):
    return all([
        (root / 'train.csv').is_file(),
        (root / 'train').is_dir(),
        (root / 'test.csv').is_file(),
        (root / 'test').is_dir(),
        (root / 'sample_submission.csv').is_file(),
    ])

def find_drive_file(filename):
    direct = DRIVE_ROOT / filename
    if direct.is_file():
        return direct
    matches = list(Path('/content/drive/MyDrive').rglob(filename))
    return matches[0] if matches else None

if has_complete_data(DRIVE_DATA_ROOT):
    DATA_ROOT = DRIVE_DATA_ROOT
else:
    DATA_ROOT = LOCAL_DATA_ROOT
    WORKDIR.mkdir(parents=True, exist_ok=True)

    if not (DATA_ROOT / 'train.csv').is_file() or not (DATA_ROOT / 'train').is_dir():
        train_bundle = find_drive_file('snu_colab_full_train_bundle.tar')
        if train_bundle is None:
            raise FileNotFoundError(
                "MyDrive에서 snu_colab_full_train_bundle.tar를 찾지 못했습니다. "
                "MyDrive/snu ai/에 업로드해 주세요."
            )
        print(f'학습 데이터 압축 해제: {train_bundle}')
        subprocess.run(['tar', '-xf', str(train_bundle), '-C', str(WORKDIR)], check=True)
        DRIVE_ROOT = train_bundle.parent

    if not all([
        (DATA_ROOT / 'test.csv').is_file(),
        (DATA_ROOT / 'test').is_dir(),
        (DATA_ROOT / 'sample_submission.csv').is_file(),
    ]):
        test_bundle = find_drive_file('snu_colab_test_bundle.tar')
        if test_bundle is None:
            raise FileNotFoundError(
                "MyDrive에서 snu_colab_test_bundle.tar를 찾지 못했습니다. "
                "MyDrive/snu ai/에 업로드해 주세요."
            )
        print(f'테스트 데이터 압축 해제: {test_bundle}')
        subprocess.run(['tar', '-xf', str(test_bundle), '-C', str(WORKDIR)], check=True)

assert has_complete_data(DATA_ROOT), f'데이터 구성이 불완전합니다: {DATA_ROOT}'
DATA_DIR = str(DATA_ROOT)
OUTPUT_DIR = str(DRIVE_ROOT / 'snu_qwen3vl_outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'DATA_DIR: {DATA_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(sorted(os.listdir(DATA_DIR)))

In [6]:
!pip install -q -U transformers accelerate peft bitsandbytes qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 219.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 54.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 101.5 MB/s eta 0:00:0000:0100:01


In [7]:
import os, re, ast, random, hashlib, time
import numpy as np
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
from tqdm.auto import tqdm
import networkx as nx
from sklearn.model_selection import StratifiedGroupKFold

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import LoraConfig, PeftModel, get_peft_model
from qwen_vl_utils import process_vision_info

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
N_FOLDS = 5
MAX_SAMPLES_PER_HASH = 3

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
train['answer_tuple'] = train['Answer'].apply(lambda x: str(ast.literal_eval(x)))

In [8]:
def compute_md5(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def get_all_image_records(df, img_dir):
    records = []
    for _, row in df.iterrows():
        for i in range(1, 5):
            path = f"{img_dir}/{row['Id']}/{row[f'Input_{i}']}"
            records.append({'sample_id': row['Id'], 'path': path})
    return records

print("Train 이미지 해시 계산 중...")
train_records = get_all_image_records(train, f'{DATA_DIR}/train')
paths = [r['path'] for r in train_records]
with ThreadPoolExecutor(max_workers=32) as executor:
    hashes = list(tqdm(executor.map(compute_md5, paths), total=len(paths)))
for r, h in zip(train_records, hashes):
    r['hash'] = h

hash_counter = defaultdict(set)
for r in train_records:
    hash_counter[r['hash']].add(r['sample_id'])

hash_to_samples = {h: s for h, s in hash_counter.items() if len(s) <= MAX_SAMPLES_PER_HASH}
excluded = sum(1 for h, s in hash_counter.items() if len(s) > MAX_SAMPLES_PER_HASH)
print(f"제외된 해시(빈번 재사용) 수: {excluded}")

G = nx.Graph()
G.add_nodes_from(train['Id'].unique())
for h, sample_ids in hash_to_samples.items():
    sample_ids = list(sample_ids)
    for i in range(len(sample_ids)):
        for j in range(i+1, len(sample_ids)):
            G.add_edge(sample_ids[i], sample_ids[j])

components = list(nx.connected_components(G))
print(f"총 {len(components)}개 클러스터, 최대 크기={max(len(c) for c in components)}")

id_to_cluster = {sid: cid for cid, comp in enumerate(components) for sid in comp}
train['cluster_id'] = train['Id'].map(id_to_cluster)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train['fold'] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(train, train['answer_tuple'], groups=train['cluster_id'])):
    train.loc[val_idx, 'fold'] = fold

cluster_fold_counts = train.groupby('cluster_id')['fold'].nunique()
print(f"여러 fold에 걸친 클러스터 수: {(cluster_fold_counts > 1).sum()} (0이어야 정상)")
print(train['fold'].value_counts().sort_index())

Train 이미지 해시 계산 중...


  0%|          | 0/38140 [00:00<?, ?it/s]

제외된 해시(빈번 재사용) 수: 2081
총 8147개 클러스터, 최대 크기=14
여러 fold에 걸친 클러스터 수: 0 (0이어야 정상)
fold
0    1915
1    1905
2    1886
3    1911
4    1918
Name: count, dtype: int64


In [9]:
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE
)

def split_clauses(sentence, max_clauses=4):
    sentence = sentence.strip()
    parts = CLAUSE_PATTERN.split(sentence)
    clauses = []
    current = parts[0].strip(' ,;.')
    for i in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[i+1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [c for c in clauses if c]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses-1] + [' '.join(clauses[max_clauses-1:])]
    if not clauses:
        clauses = [sentence]
    return clauses

In [10]:
STRUCTURED_HINT = (
    "This caption describes distinct sequential stages, often marked by words like "
    "'then', 'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    "This caption describes a single continuous action or a brief moment. "
    "Focus on subtle visual cues such as camera movement direction, object position changes, "
    "hand/body motion, and lighting changes to infer the temporal order."
)

def get_prompt_message(row, image_dir):
    img_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
    sentence = row['Sentence']
    clause_count = len(split_clauses(sentence))
    hint = STRUCTURED_HINT if clause_count >= 3 else CONTINUOUS_HINT

    content = []
    for i, img_file in enumerate(img_files):
        img_path = os.path.join(image_dir, row['Id'], img_file)
        content.append({"type": "image", "image": img_path})
        content.append({"type": "text", "text": f"\nImage {i+1}\n"})

    user_text = (
        f"Thinking about the sentence: \"{sentence}\"\n"
        f"{hint}\n"
        "Look at the 4 images above labeled Image 1 to Image 4. "
        "Determine the correct chronological order of these images to match the sentence. "
        "Provide the answer ONLY as a Python list of integers. "
        "Example: [1, 2, 3, 4]"
    )
    content.append({"type": "text", "text": user_text})
    return [{"role": "user", "content": content}]


def format_training_sample(row, img_dir):
    messages = get_prompt_message(row, img_dir)
    answer = ast.literal_eval(row['Answer'])
    temporal_order = [0, 0, 0, 0]
    for input_idx, pos in enumerate(answer):
        temporal_order[pos - 1] = input_idx + 1
    target_text = str(temporal_order)
    messages.append({"role": "assistant", "content": [{"type": "text", "text": target_text}]})
    return messages, target_text


def parse_model_output(output_text):
    try:
        start_idx = output_text.find('[')
        end_idx = output_text.rfind(']')
        if start_idx != -1 and end_idx != -1:
            result = ast.literal_eval(output_text[start_idx:end_idx+1])
            if isinstance(result, list) and len(result) == 4 and sorted(result) == [1,2,3,4]:
                submission_answer = [0]*4
                for index, image_num in enumerate(result):
                    submission_answer[image_num - 1] = index + 1
                return submission_answer, True
    except:
        pass
    return [1,2,3,4], False

In [11]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.1 MB/s eta 0:00:00:00:01


In [12]:
import shutil
cache_path = os.path.expanduser('~/.cache/huggingface/hub')
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("HF 캐시 삭제 완료, 재다운로드 필요")

In [13]:
import shutil, os
for p in ['~/.cache/huggingface', '/content/qwen3vl_local']:
    full_path = os.path.expanduser(p)
    if os.path.exists(full_path):
        shutil.rmtree(full_path)
        print(f"삭제: {full_path}")

# ⚠️ 복구 전용: Epoch 4 validation부터 판정 재개

Epoch 4 학습은 완료됐지만 validation 판정 직후 런타임이 끊겼을 가능성이 있으므로, 아래 셀들이 Epoch 4를 다시 validation한 뒤 Epoch 5 학습 여부를 자동으로 결정합니다.

아래 코드 셀 3개를 순서대로 실행하세요.

1. `# 같은 런타임에서 fresh 학습을 중단한 뒤에도 안전하게 resume...`로 시작하는 셀
2. `class QwenOrderingDataset(Dataset):`로 시작하는 셀
3. `MAX_EPOCHS = 5`와 `Recovery comparison`이 들어 있는 복구·학습 셀

`ALLOW_TRAINING` 수동 플래그는 사용하지 않습니다. 세 번째 셀은 Drive에서 찾은 마지막 완결 epoch adapter를 기준으로 복구 상태를 판정합니다.

세 번째 셀은 기존 best인 `qwen3vl_final`과 마지막 완결 epoch adapter의 validation loss를 다시 비교합니다.

- Epoch 4가 개선됐으면 best를 갱신하고 **Epoch 5/5를 학습**합니다.
- Epoch 4가 개선되지 않았으면 비개선 2회로 **early stopping**하고 Epoch 5를 건너뜁니다.

세 번째 셀 시작 시 출력되는 `학습 GPU:`가 Colab GPU 이름인지 확인하세요. CUDA가 없으면 CPU 학습으로 넘어가지 않고 즉시 중단됩니다.

판정과 필요 학습이 끝난 뒤 `✅ 최종 best 평가` 안내로 이동하세요.

In [17]:
# 같은 런타임에서 fresh 학습을 중단한 뒤에도 안전하게 resume할 수 있게 정리
import gc
for variable_name in (
    'optimizer', 'outputs', 'raw_loss', 'loss', 'batch', 'model', 'base_model'
):
    globals().pop(variable_name, None)
gc.collect()
torch.cuda.empty_cache()

# 베이스 모델을 다시 로드한 뒤 저장된 완결 epoch adapter를 결합한다.
processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

def find_latest_completed_adapter(output_dir):
    candidates = []
    for path in Path(output_dir).glob('qwen3vl_epoch*'):
        suffix = path.name.removeprefix('qwen3vl_epoch')
        if not suffix.isdigit():
            continue
        has_config = (path / 'adapter_config.json').is_file()
        has_weights = any([
            (path / 'adapter_model.safetensors').is_file(),
            (path / 'adapter_model.bin').is_file(),
        ])
        if not (has_config and has_weights):
            continue
        # 예전 코드는 첫 epoch를 epoch0으로, 새 코드는 epoch1로 저장한다.
        completed_epochs = max(1, int(suffix))
        candidates.append((completed_epochs, path.stat().st_mtime, path))
    if not candidates:
        return None, 0
    completed_epochs, _, path = max(candidates)
    return path, completed_epochs

RESUME_ADAPTER_DIR, COMPLETED_EPOCHS = find_latest_completed_adapter(OUTPUT_DIR)
if RESUME_ADAPTER_DIR is not None:
    model = PeftModel.from_pretrained(
        model,
        str(RESUME_ADAPTER_DIR),
        is_trainable=True,
    )
    print(
        f'완료된 {COMPLETED_EPOCHS} epoch adapter에서 재개: '
        f'{RESUME_ADAPTER_DIR}'
    )
else:
    # 저장된 완결 epoch가 없을 때만 새 LoRA adapter를 만든다.
    attn_module_names = set()
    for name, _ in model.named_modules():
        if name.split('.')[-1] in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
            attn_module_names.add(name.split('.')[-1])
    print('발견된 attention 모듈:', attn_module_names)

    lora_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=list(attn_module_names) if attn_module_names else ['q_proj', 'v_proj'],
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

완료된 4 epoch adapter에서 재개: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_epoch4
trainable params: 15,335,424 || all params: 8,782,459,120 || trainable%: 0.1746


In [18]:
class QwenOrderingDataset(Dataset):
    def __init__(self, df, img_dir, processor):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        messages, target_text = format_training_sample(row, self.img_dir)

        full_text = self.processor.apply_chat_template(messages, tokenize=False)
        prompt_only = self.processor.apply_chat_template(messages[:1], tokenize=False, add_generation_prompt=True)

        image_inputs, _ = process_vision_info(messages)

        full_enc = self.processor(text=[full_text], images=image_inputs, return_tensors="pt", padding=False)
        prompt_enc = self.processor(text=[prompt_only], images=image_inputs, return_tensors="pt", padding=False)

        input_ids = full_enc['input_ids'][0]
        labels = input_ids.clone()
        prompt_len = prompt_enc['input_ids'].shape[1]
        labels[:prompt_len] = -100

        return {
            'input_ids': input_ids,
            'attention_mask': full_enc['attention_mask'][0],
            'labels': labels,
            'pixel_values': full_enc['pixel_values'],
            'image_grid_thw': full_enc['image_grid_thw'],
            'mm_token_type_ids': full_enc['mm_token_type_ids'][0],
        }


def collate_fn(batch):
    max_len = max(item['input_ids'].shape[0] for item in batch)
    pad_id = processor.tokenizer.pad_token_id

    input_ids, attn_mask, labels, mm_token_type_ids = [], [], [], []
    for item in batch:
        seq_len = item['input_ids'].shape[0]
        pad_len = max_len - seq_len
        input_ids.append(torch.cat([item['input_ids'], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        attn_mask.append(torch.cat([item['attention_mask'], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([item['labels'], torch.full((pad_len,), -100, dtype=torch.long)]))
        mm_token_type_ids.append(torch.cat([item['mm_token_type_ids'], torch.zeros(pad_len, dtype=torch.long)]))

    pixel_values = torch.cat([item['pixel_values'] for item in batch], dim=0)
    image_grid_thw = torch.cat([item['image_grid_thw'] for item in batch], dim=0)

    return {
        'input_ids': torch.stack(input_ids),
        'attention_mask': torch.stack(attn_mask),
        'labels': torch.stack(labels),
        'pixel_values': pixel_values,
        'image_grid_thw': image_grid_thw,
        'mm_token_type_ids': torch.stack(mm_token_type_ids),
    }

In [19]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU가 없습니다. Colab 런타임을 GPU로 설정한 뒤 다시 실행하세요.')
print(f'학습 GPU: {torch.cuda.get_device_name(0)}')
torch.cuda.empty_cache()

# 모델에 gradient checkpointing 적용
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False

def safe_save_adapter(model, path):
    tmp_path = path + '_tmp'
    if os.path.exists(tmp_path):
        shutil.rmtree(tmp_path)
    model.save_pretrained(tmp_path, selected_adapters=['default'])
    if os.path.exists(path):
        shutil.rmtree(path)
    os.rename(tmp_path, path)
    print(f"저장 완료: {path}")

fold = 0
tr_df = train[train['fold'] != fold].reset_index(drop=True)
va_df = train[train['fold'] == fold].reset_index(drop=True)
print(f"Train: {len(tr_df)}, Val: {len(va_df)}")

tr_ds = QwenOrderingDataset(tr_df, f'{DATA_DIR}/train', processor)
va_ds = QwenOrderingDataset(va_df, f'{DATA_DIR}/train', processor)
tr_loader = DataLoader(tr_ds, batch_size=1, shuffle=True, collate_fn=collate_fn, num_workers=2)  # A100이면 2~4 시도 가능
va_loader = DataLoader(va_ds, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=2)

import bitsandbytes as bnb
optimizer = bnb.optim.AdamW8bit(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)
GRAD_ACCUM_STEPS = 8
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-4
MAX_GRAD_NORM = 1.0
CHECKPOINT_EVERY = 500
BEST_ADAPTER_DIR = f'{OUTPUT_DIR}/qwen3vl_best'
FINAL_ADAPTER_DIR = f'{OUTPUT_DIR}/qwen3vl_final'
if 'COMPLETED_EPOCHS' not in globals():
    raise RuntimeError('resume 상태를 정한 모델 로드 셀을 먼저 실행하세요.')
if COMPLETED_EPOCHS not in (4, 5):
    raise RuntimeError(
        f'Epoch 4 또는 Epoch 5 완결 adapter가 필요하지만 완료 epoch={COMPLETED_EPOCHS}입니다.'
    )
if not Path(FINAL_ADAPTER_DIR).is_dir():
    raise FileNotFoundError(FINAL_ADAPTER_DIR)

def evaluate_validation_loss(model, loader):
    model.eval()
    loss_sum = 0.0
    sample_count = 0
    with torch.inference_mode():
        for batch in tqdm(loader, desc='Validation loss', leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            val_loss = model(**batch).loss
            if not torch.isfinite(val_loss):
                raise RuntimeError(f'비정상 validation loss: {val_loss.item()}')
            batch_size = batch['input_ids'].shape[0]
            loss_sum += val_loss.item() * batch_size
            sample_count += batch_size
    return loss_sum / sample_count

# Epoch 4 복구 시 기존 best는 Epoch 2이고 비개선 1회 상태다.
# Epoch 5 adapter가 있으면 Epoch 4가 개선되어 Epoch 5 학습까지 완료된 상태다.
if COMPLETED_EPOCHS == 4:
    REFERENCE_BEST_EPOCH = 2
    PREVIOUS_EPOCHS_WITHOUT_IMPROVEMENT = 1
else:
    REFERENCE_BEST_EPOCH = 4
    PREVIOUS_EPOCHS_WITHOUT_IMPROVEMENT = 0
REFERENCE_ADAPTER_NAME = 'best_reference'
if REFERENCE_ADAPTER_NAME in model.peft_config:
    raise RuntimeError('모델 로드 셀을 다시 실행한 뒤 복구 셀을 실행하세요.')
model.load_adapter(
    FINAL_ADAPTER_DIR,
    adapter_name=REFERENCE_ADAPTER_NAME,
    is_trainable=False,
)

# 동일한 validation fold에서 기존 best와 마지막 완결 epoch를 다시 비교한다.
model.set_adapter(REFERENCE_ADAPTER_NAME, inference_mode=True)
best_val_loss = evaluate_validation_loss(model, va_loader)
best_epoch = REFERENCE_BEST_EPOCH

model.set_adapter('default', inference_mode=True)
resumed_val_loss = evaluate_validation_loss(model, va_loader)
model.set_adapter('default', inference_mode=False)

epochs_without_improvement = PREVIOUS_EPOCHS_WITHOUT_IMPROVEMENT
training_history = [
    {'epoch': best_epoch, 'train_loss': None, 'val_loss': best_val_loss, 'reference_best': True},
    {'epoch': COMPLETED_EPOCHS, 'train_loss': None, 'val_loss': resumed_val_loss, 'resumed': True},
]
print(
    f"Recovery comparison: best_epoch={best_epoch}, "
    f"best_val_loss={best_val_loss:.4f}, "
    f"epoch{COMPLETED_EPOCHS}_val_loss={resumed_val_loss:.4f}"
)

continue_training = True
if resumed_val_loss < best_val_loss - EARLY_STOPPING_MIN_DELTA:
    best_val_loss = resumed_val_loss
    best_epoch = COMPLETED_EPOCHS
    epochs_without_improvement = 0
    safe_save_adapter(model, BEST_ADAPTER_DIR)
    safe_save_adapter(model, FINAL_ADAPTER_DIR)
    print(f"Epoch {COMPLETED_EPOCHS}가 새 best입니다: val_loss={best_val_loss:.4f}")
else:
    epochs_without_improvement += 1
    print(
        f"Epoch {COMPLETED_EPOCHS} 개선 없음: {epochs_without_improvement}/"
        f"{EARLY_STOPPING_PATIENCE}"
    )
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        continue_training = False
        print(f"Early stopping after Epoch {COMPLETED_EPOCHS}; best epoch={best_epoch}")

epochs_to_run = range(COMPLETED_EPOCHS, MAX_EPOCHS) if continue_training else range(0)
for epoch in epochs_to_run:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    for step, batch in enumerate(tqdm(tr_loader, desc=f"Epoch {epoch + 1}/{MAX_EPOCHS}")):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        raw_loss = outputs.loss
        if not torch.isfinite(raw_loss):
            raise RuntimeError(f'비정상 train loss: step={step}, loss={raw_loss.item()}')
        (raw_loss / GRAD_ACCUM_STEPS).backward()

        is_update_step = ((step + 1) % GRAD_ACCUM_STEPS == 0) or (step + 1 == len(tr_loader))
        if is_update_step:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += raw_loss.item()
        if step % 100 == 0:
            print(f" step {step}: loss={raw_loss.item():.4f}")

        if (step + 1) % CHECKPOINT_EVERY == 0:
            safe_save_adapter(model, f'{OUTPUT_DIR}/qwen3vl_checkpoint_latest')

    train_loss = total_loss / len(tr_loader)
    # validation 중 런타임이 끊겨도 학습 완료 adapter는 Drive에 남긴다.
    safe_save_adapter(model, f'{OUTPUT_DIR}/qwen3vl_epoch{epoch + 1}')
    val_loss = evaluate_validation_loss(model, va_loader)
    training_history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
    })
    print(f"Epoch {epoch + 1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    if val_loss < best_val_loss - EARLY_STOPPING_MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        safe_save_adapter(model, BEST_ADAPTER_DIR)
        safe_save_adapter(model, FINAL_ADAPTER_DIR)
        print(f"새 best adapter: epoch={best_epoch}, val_loss={best_val_loss:.4f}")
    else:
        epochs_without_improvement += 1
        print(
            f"개선 없음: {epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch + 1}; best epoch={best_epoch}")
            break

print(f"Best epoch={best_epoch}, best validation loss={best_val_loss:.4f}")
print(f"Best adapter path: {FINAL_ADAPTER_DIR}")

학습 GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Train: 7620, Val: 1915


Validation loss:   0%|          | 0/1915 [00:00<?, ?it/s]

Validation loss:   0%|          | 0/1915 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a18e0280540>
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Traceback (most recent call last):
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a18e0280540>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Recovery comparison: best_epoch=2, best_val_loss=0.1117, epoch4_val_loss=0.1224
Epoch 4 개선 없음: 2/2
Early stopping after Epoch 4; best epoch=2
Best epoch=2, best validation loss=0.1117
Best adapter path: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_final


# ✅ 학습 완료 adapter로 validation CSV 생성

학습이 이미 끝났다면 위의 복구·학습 셀은 실행하지 않아도 됩니다. Drive에 저장된 `qwen3vl_final` adapter를 불러와 fold 0 validation 추론과 CSV 저장만 수행합니다.

새 Colab 런타임에서는 위쪽 준비 셀 중 다음 항목까지만 실행하세요.

- Google Drive 마운트와 `DATA_DIR` / `OUTPUT_DIR` 설정
- 패키지 설치 및 공통 import
- `train.csv` 로드와 MD5 group fold 생성
- `split_clauses`, `get_prompt_message`, `parse_model_output` 함수 정의

그 다음 바로 아래 코드 셀 2개만 순서대로 실행하세요.

1. `import gc`로 시작하는 **best adapter 로드 셀**
2. `from scipy.stats import kendalltau`로 시작하는 **validation 추론·CSV 다운로드 셀**

두 번째 셀은 상세 CSV를 Drive의 `snu_qwen3vl_outputs/validation_predictions_qwen3vl8b_lora_fold0.csv`에 저장하고 브라우저 다운로드를 시작합니다. 그 아래 test inference 셀들은 실행하지 않아도 됩니다.

In [20]:
import gc
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

# 새 런타임에서 바로 실행 가능한 final evaluation 진입점
processor = AutoProcessor.from_pretrained(MODEL_NAME)
FINAL_ADAPTER_DIR = Path(OUTPUT_DIR) / 'qwen3vl_final'
assert (FINAL_ADAPTER_DIR / 'adapter_config.json').is_file(), FINAL_ADAPTER_DIR
assert any([
    (FINAL_ADAPTER_DIR / 'adapter_model.safetensors').is_file(),
    (FINAL_ADAPTER_DIR / 'adapter_model.bin').is_file(),
]), FINAL_ADAPTER_DIR

# 기존 model이 있다면 정리
try:
    del model
except NameError:
    pass
try:
    del base_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# 1. 순수 베이스 모델 새로 로드
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
)

# 2. validation 기준 best adapter를 명시적으로 로드
model = PeftModel.from_pretrained(base_model, str(FINAL_ADAPTER_DIR))
model.eval()
print(f'Best adapter loaded: {FINAL_ADAPTER_DIR}')

# 3. 확인 - 0이 아니어야 정상
for name, param in model.named_parameters():
    if 'lora_B' in name:
        print(f"LoRA B 값 확인: {param.abs().mean().item()}")
        break

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Best adapter loaded: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_final
LoRA B 값 확인: 0.008305066265165806


In [ ]:
from scipy.stats import kendalltau

if 'fold' not in train.columns:
    raise RuntimeError('먼저 MD5 group fold 생성 셀을 실행하세요.')
fold = 0
va_df = train[train['fold'] == fold].reset_index(drop=True)
print(f'Validation samples: {len(va_df)}')

def compute_metrics(pred, target):
    exact_match = (pred == target).all(axis=1).mean()
    position_acc = (pred == target).mean()
    taus = [kendalltau(p,t)[0] if not np.isnan(kendalltau(p,t)[0]) else 1.0 for p,t in zip(pred,target)]
    return {'exact_match': exact_match, 'position_acc': position_acc, 'kendall_tau': np.mean(taus)}

processor.tokenizer.padding_side = 'left'  # 배치 생성을 위해 필수

BATCH_SIZE = 8

model.eval()
results = []

for start_idx in tqdm(range(0, len(va_df), BATCH_SIZE)):
    batch_rows = va_df.iloc[start_idx:start_idx + BATCH_SIZE]

    batch_messages = [get_prompt_message(row, f'{DATA_DIR}/train') for _, row in batch_rows.iterrows()]
    batch_texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch_messages]

    all_image_inputs = []
    for m in batch_messages:
        imgs, _ = process_vision_info(m)
        all_image_inputs.extend(imgs)

    inputs = processor(
        text=batch_texts, images=all_image_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    output_texts = processor.batch_decode(trimmed, skip_special_tokens=True)

    for (_, row), output_text in zip(batch_rows.iterrows(), output_texts):
        pred, parse_ok = parse_model_output(output_text)
        results.append({
            'Id': row['Id'],
            'pred': pred,
            'parse_ok': parse_ok,
            'raw_output': output_text,
        })

assert len(results) == len(va_df), (len(results), len(va_df))
assert [r['Id'] for r in results] == va_df['Id'].tolist(), 'validation Id 순서가 달라졌습니다.'

pred_array = np.array([r['pred'] for r in results]) - 1
target_array = np.array([ast.literal_eval(a) for a in va_df['Answer']]) - 1

overall = compute_metrics(pred_array, target_array)
no_order_mask = va_df['No_ordering'].values
shuffled = compute_metrics(pred_array[~no_order_mask], target_array[~no_order_mask])

print(f"[Qwen3-VL-8B + 조건부 프롬프트, Group-CV val, n={len(va_df)}]")
print(f"  전체 exact={overall['exact_match']:.4f}")
print(f"  셔플됨 exact={shuffled['exact_match']:.4f}")
print(f"  파싱 실패율={1-np.mean([r['parse_ok'] for r in results]):.1%}")

validation_predictions_df = pd.DataFrame({
    'Id': va_df['Id'],
    'Fold': fold,
    'Answer': va_df['Answer'],
    'Prediction': [str(r['pred']) for r in results],
    'Correct': (pred_array == target_array).all(axis=1),
    'ParseOk': [r['parse_ok'] for r in results],
    'No_ordering': va_df['No_ordering'],
    'RawOutput': [r['raw_output'] for r in results],
})

validation_csv_path = Path(OUTPUT_DIR) / 'validation_predictions_qwen3vl8b_lora_fold0.csv'
validation_predictions_df.to_csv(validation_csv_path, index=False, encoding='utf-8-sig')

saved_validation_df = pd.read_csv(validation_csv_path)
assert len(saved_validation_df) == len(va_df)
assert saved_validation_df['Id'].tolist() == va_df['Id'].tolist()
assert saved_validation_df['Prediction'].notna().all()

print(f'Validation CSV 저장 완료: {validation_csv_path}')
print(f'CSV rows: {len(saved_validation_df)}, columns: {list(saved_validation_df.columns)}')
display(validation_predictions_df.head())

from google.colab import files
files.download(str(validation_csv_path))

In [22]:
test = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f"Test samples: {len(test)}")

Test samples: 819


In [23]:
import gc

torch.cuda.empty_cache()
gc.collect()

processor.tokenizer.padding_side = 'left'
BATCH_SIZE = 4
TEST_IMAGE_DIR = f'{DATA_DIR}/test'

# 이미 처리된 결과가 있으면 이어서 (재시작 대비)
if 'test_results' not in dir():
    test_results = []

processed_ids = set(r['Id'] for r in test_results)
remaining_test = test[~test['Id'].isin(processed_ids)].reset_index(drop=True)
print(f"남은 샘플 수: {len(remaining_test)} / {len(test)}")

model.eval()
for start_idx in tqdm(range(0, len(remaining_test), BATCH_SIZE)):
    batch_rows = remaining_test.iloc[start_idx:start_idx + BATCH_SIZE]

    batch_messages = [get_prompt_message(row, TEST_IMAGE_DIR) for _, row in batch_rows.iterrows()]
    batch_texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch_messages]

    all_image_inputs = []
    for m in batch_messages:
        imgs, _ = process_vision_info(m)
        all_image_inputs.extend(imgs)

    inputs = processor(
        text=batch_texts, images=all_image_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)

    try:
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    except torch.cuda.OutOfMemoryError:
        print(f"OOM 발생, 이 배치는 배치사이즈 1로 재시도")
        torch.cuda.empty_cache()
        # 배치를 1개씩 쪼개서 재시도
        generated_ids_list = []
        for i in range(len(batch_rows)):
            single_input = {k: v[i:i+1] for k, v in inputs.items()}
            with torch.no_grad():
                gid = model.generate(**single_input, max_new_tokens=64, do_sample=False)
            generated_ids_list.append(gid[0])
        # 길이 맞춰 패딩 후 스택 (간단히 개별 디코딩으로 처리)
        trimmed = []
        for i, gid in enumerate(generated_ids_list):
            inp_len = (inputs['input_ids'][i] != processor.tokenizer.pad_token_id).sum().item()
            trimmed.append(gid[len(inputs['input_ids'][i]):])
        output_texts = [processor.tokenizer.decode(t, skip_special_tokens=True) for t in trimmed]
        for (_, row), output_text in zip(batch_rows.iterrows(), output_texts):
            pred, parse_ok = parse_model_output(output_text)
            test_results.append({'Id': row['Id'], 'pred': pred, 'parse_ok': parse_ok})
        continue

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    output_texts = processor.batch_decode(trimmed, skip_special_tokens=True)

    for (_, row), output_text in zip(batch_rows.iterrows(), output_texts):
        pred, parse_ok = parse_model_output(output_text)
        test_results.append({'Id': row['Id'], 'pred': pred, 'parse_ok': parse_ok})

print(f"완료: {len(test_results)} / {len(test)}")
print(f"파싱 실패율: {1 - np.mean([r['parse_ok'] for r in test_results]):.1%}")

남은 샘플 수: 819 / 819


  0%|          | 0/205 [00:00<?, ?it/s]

완료: 819 / 819
파싱 실패율: 0.0%


In [24]:
print(f"지금까지 처리된 개수: {len(test_results)}")

지금까지 처리된 개수: 819


In [25]:
predictions = [{'Id': r['Id'], 'Answer': str(r['pred'])} for r in test_results]
submission_df = pd.DataFrame(predictions)

submit_path = f'{OUTPUT_DIR}/submission_qwen3vl8b.csv'
submission_df.to_csv(submit_path, index=False)

# 형식 검증
sample_submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
assert list(submission_df.columns) == list(sample_submission.columns), "컬럼명 불일치!"
assert len(submission_df) == len(sample_submission), "행 개수 불일치!"
assert set(submission_df['Id']) == set(sample_submission['Id']), "Id 불일치!"

print(f"저장 완료: {submit_path}")
print("형식 검증 통과 — 제출 가능합니다.")
submission_df.head()

저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/submission_qwen3vl8b.csv
형식 검증 통과 — 제출 가능합니다.


,Id,Answer
0,WI8o3F,"[1, 4, 2, 3]"
1,Ae7zLm,"[2, 3, 4, 1]"
2,4QSIII,"[1, 4, 2, 3]"
3,k7h4gt,"[4, 1, 2, 3]"
4,ZtTwCx,"[1, 3, 2, 4]"


In [26]:
from collections import Counter

pred_strs = [str(r['pred']) for r in test_results]
print("전체 예측 분포 상위 5개:", Counter(pred_strs).most_common(5))

전체 예측 분포 상위 5개: [('[1, 2, 3, 4]', 119), ('[4, 2, 3, 1]', 44), ('[1, 3, 4, 2]', 42), ('[3, 2, 4, 1]', 41), ('[1, 2, 4, 3]', 39)]
